In [1]:
import pandas as pd
import numpy as np
import os

# Load processed signal data (daily raw + weekday raw)
DATA_FOLDER = "Signal_output_hourly" 
hourly_raw_path = os.path.join(DATA_FOLDER, "signal_hourly_combined.csv")
meta_path = os.path.join(DATA_FOLDER, "signal_metadata_split.csv")

hourly_df = pd.read_csv(hourly_raw_path)

print("Hourly shape:", hourly_df.shape)

Hourly shape: (1195806, 46)


In [2]:
# Compute turning ratios for each signal using all directions
def compute_overall_turn_ratios(df):
    df = df.copy()

    # All L/T/R columns after TL/TR splitting
    left_cols = [c for c in df.columns if c.startswith("Vehicle_") and c.endswith("_L")]
    through_cols = [c for c in df.columns if c.startswith("Vehicle_") and c.endswith("_T")]
    right_cols = [c for c in df.columns if c.startswith("Vehicle_") and c.endswith("_R")]

    # Sum across all approaches for each row
    df["left_total"] = df[left_cols].sum(axis=1)
    df["through_total"] = df[through_cols].sum(axis=1)
    df["right_total"] = df[right_cols].sum(axis=1)

    df["turn_total"] = df["left_total"] + df["through_total"] + df["right_total"]

    # Compute ratios
    df["left_ratio"] = np.where(df["turn_total"] > 0,
                                df["left_total"] / df["turn_total"], np.nan)
    df["through_ratio"] = np.where(df["turn_total"] > 0,
                                   df["through_total"] / df["turn_total"], np.nan)
    df["right_ratio"] = np.where(df["turn_total"] > 0,
                                 df["right_total"] / df["turn_total"], np.nan)

    return df


hourly_ratio = compute_overall_turn_ratios(hourly_df)

# Keep only rows with actual turning volume
hourly_ratio = hourly_ratio[hourly_ratio["turn_total"] > 0].copy()

print("Remaining signals in hourly_ratio:", hourly_ratio["SignalID"].nunique())


Remaining signals in hourly_ratio: 264


In [3]:
# Preview hourly turn ratios
hourly_ratio[[
    "SignalID", "Datetime", "hour", "dow", "dow_name",
    "intersection_type",
    "left_ratio", "through_ratio", "right_ratio"
]].head()

,SignalID,Datetime,hour,dow,dow_name,intersection_type,left_ratio,through_ratio,right_ratio
0,6129022,2025-05-01 00:00:00,0.0,3.0,Thursday,4_way,0.268182,0.731818,0.0
1,6129022,2025-05-01 01:00:00,1.0,3.0,Thursday,4_way,0.166667,0.833333,0.0
2,6129022,2025-05-01 02:00:00,2.0,3.0,Thursday,4_way,0.220000,0.780000,0.0
3,6129022,2025-05-01 03:00:00,3.0,3.0,Thursday,4_way,0.281818,0.718182,0.0
4,6129022,2025-05-01 04:00:00,4.0,3.0,Thursday,4_way,0.316327,0.683673,0.0


In [4]:
ratio_output = hourly_ratio[[
    "SignalID", "Datetime", "hour", "date", "dow", "dow_name", "intersection_type",
    "left_ratio", "through_ratio", "right_ratio"
]]

output_path = os.path.join(DATA_FOLDER, "signal_hourly_ratio.csv")
ratio_output.to_csv(output_path, index=False)

print("Saved hourly ratio to:", output_path)


Saved hourly ratio to: Signal_output_hourly/signal_hourly_ratio.csv
